# Notebook: Cálculo do Índice de Vulnerabilidade Social de Hortolândia (IVS-H)

**id_rastreabilidade:** RTB_005  
**Versão:** v02  
**Data de criação:** 20/04/2026  
**Última atualização:** 08/05/2026  
**Responsável:** Ailton Vendramini  

---

## 🎯 Finalidade
Calcular indicadores de vulnerabilidade social a partir da base limpa do CadÚnico, com foco em:
1. estrutura de cálculo alinhada ao IVS de referência;
2. adaptação metodológica para o contexto local de Hortolândia (IVS-H);
3. produção de saídas analíticas utilizáveis em diagnóstico e formulação de políticas públicas.

---

## 📥 Fonte de Dados
**Fonte:** CadÚnico  
**Camada de entrada:** `02_limpo`  
**Camada de saída:** `03_curado` ou `04_exportacao`  
**Período de referência da execução:** `2025_12`  
**Período de comparação:** —  
**Tipo de recorte temporal:** corte transversal  

**Arquivo de entrada:**  
`C:\\Users\\ailto\\Atlas-Social-de-Hortolandia\\dados\\cadunico\\02_limpo\\2025_12\\cadunico_limpo.csv`

---

## 🧱 Base Conceitual
- `docs/metodologia_ivsh.md`
- `docs/regras_negocio.md`
- `docs/dicionario.md`
- `00_governanca/ivs_vs_ivsh_comparativo_v09.md`

---

## ⚙️ Escopo técnico deste notebook
Este notebook deve:
- carregar a base limpa do CadÚnico;
- validar colunas necessárias ao cálculo;
- aplicar regras de negócio e fórmulas dos indicadores;
- gerar tabelas intermediárias de conferência;
- produzir saídas consolidadas para uso analítico posterior.

---

## 📊 Outputs esperados

| tipo_output | descrição | destino_previsto |
|---|---|---|
| analitico | tabelas intermediárias de cálculo e conferência | `outputs/tabelas/cadunico/` |
| curado | base com indicadores derivados | `dados/cadunico/03_curado/2025_12/` |
| exportacao | resultados consolidados para uso executivo | `dados/cadunico/04_exportacao/2025_12/` |

---

## 🧠 Observações
- O período de referência deve ser explicitado também na primeira célula de código.
- Alterações estruturais relevantes devem ser registradas em nota técnica futura.
- Este notebook não deve reproduzir etapas de limpeza já executadas no `02_tratamento_cadunico.ipynb`.
- Toda adaptação metodológica em relação ao IVS de referência deve ser explicitada de forma rastreável.
- Unidade de análise: **RT_01 e RT_04** → família | **CH_06** → pessoa

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import date

## 2. Carregamento da base limpa

In [ ]:
# constantes de referência — usadas em todo o notebook
PERIODO_REFERENCIA = '2025_12'
DATA_REFERENCIA = date(2025, 12, 31)
MEIO_SM = 759.00  # meio salário mínimo em dez/2025

# caminho absoluto da base limpa gerada pelo notebook 02
caminho_entrada = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\cadunico\02_limpo\2025_12\cadunico_limpo.csv'

# sep=';'            → padrão exportado pelo notebook 02
# encoding='utf-8'   → padrão exportado pelo notebook 02
# low_memory=False   → evita inferência parcial de tipos
df = pd.read_csv(caminho_entrada, sep=';', encoding='utf-8', low_memory=False)

print(f'Base carregada: {df.shape[0]} linhas, {df.shape[1]} colunas')
print(f'Período de referência: {PERIODO_REFERENCIA}')

## 3. Validação de colunas necessárias

In [ ]:
# colunas mínimas para calcular as variáveis da Fase 1 MVP
colunas_necessarias = [
    'd.vlr_renda_media_fam',       # RT_01 e RT_04
    'd.cod_familiar_fam',           # chave de família
    'idade',                        # RT_04 e CH_06 — criada no notebook 02
    'p.cod_sabe_ler_escrever_memb', # CH_06
    'p.cod_sexo_pessoa',            # CH_06 recorte de gênero
    'p.ind_frequenta_escola_memb',  # CH_06 trajetória escolar
    'd.nom_localidade_fam',         # CH_06 proxy territorial
    'd.marc_pbf'                    # CH_06 cruzamento BPC/BF
]

ausentes = [col for col in colunas_necessarias if col not in df.columns]

if ausentes:
    print(f'ATENÇÃO — colunas ausentes: {ausentes}')
else:
    print('Todas as colunas necessárias estão presentes')

## 4. RT_01 — Renda per capita ≤ ½ SM

In [ ]:
# RT_01: % de famílias com renda per capita ≤ ½ SM
# unidade de análise: FAMÍLIA (não pessoa)
# deduplicamos por d.cod_familiar_fam para não contar a mesma família múltiplas vezes

# uma linha por família (first preserva o valor do campo familiar)
df_familias = df.drop_duplicates(subset='d.cod_familiar_fam')

total_familias = len(df_familias)
familias_rt01 = (df_familias['d.vlr_renda_media_fam'] <= MEIO_SM).sum()

rt01 = (familias_rt01 / total_familias * 100).round(2)

print(f'Total de famílias        : {total_familias}')
print(f'Famílias com renda ≤ ½ SM: {familias_rt01}')
print(f'RT_01                    : {rt01}%')

# complemento: famílias com renda declarada igual a zero
familias_renda_zero = (df_familias['d.vlr_renda_media_fam'] == 0).sum()
print(f'Famílias com renda zero  : {familias_renda_zero}')

## 5. RT_04 — Renda ≤ ½ SM + presença de idoso (≥ 60 anos)

In [ ]:
# RT_04: % de famílias com renda per capita ≤ ½ SM E pelo menos um membro com 60+
# unidade de análise: FAMÍLIA

# identificar famílias que têm pelo menos um idoso
familias_com_idoso = df[df['idade'] >= 60]['d.cod_familiar_fam'].unique()

# famílias com idoso E renda ≤ ½ SM
mask_rt04 = (
    (df_familias['d.cod_familiar_fam'].isin(familias_com_idoso)) &
    (df_familias['d.vlr_renda_media_fam'] <= MEIO_SM)
)

familias_rt04 = mask_rt04.sum()
rt04 = (familias_rt04 / total_familias * 100).round(2)

print(f'Famílias RT_04 (renda ≤ ½ SM + idoso): {familias_rt04}')
print(f'RT_04                                 : {rt04}%')

## 6. CH_06 — Taxa de analfabetismo (15 anos ou mais)

In [ ]:
# CH_06: % de pessoas com 15+ que não sabem ler e escrever
# unidade de análise: PESSOA
# fonte: p.cod_sabe_ler_escrever_memb == 2 (não sabe)

# masks base
mask_15mais    = df['idade'] >= 15                                    # pessoas com 15+
mask_analfabeto = df['p.cod_sabe_ler_escrever_memb'] == 2             # não sabe ler
mask_ch06      = mask_15mais & mask_analfabeto                        # interseção

total_15mais   = mask_15mais.sum()
total_ch06     = mask_ch06.sum()
ch06           = (total_ch06 / total_15mais * 100).round(2)

print(f'Total pessoas 15+    : {total_15mais}')
print(f'Analfabetos 15+      : {total_ch06}')
print(f'CH_06 municipal      : {ch06}%')

In [ ]:
# recorte etário: faixa produtiva (15–59) vs idosos (60+)
mask_15_59     = (df['idade'] >= 15) & (df['idade'] < 60)
mask_60mais    = df['idade'] >= 60

ch06_prod      = (mask_15_59 & mask_analfabeto)
ch06_idosos    = (mask_60mais & mask_analfabeto)

print(f'Analfabetos 15–59 (produtiva) : {ch06_prod.sum()}  ({(ch06_prod.sum()/mask_15_59.sum()*100).round(2)}%)')
print(f'Analfabetos 60+  (idosos)     : {ch06_idosos.sum()}')

In [ ]:
# recorte de gênero
# p.cod_sexo_pessoa: 1=masculino, 2=feminino
mask_feminino  = df['p.cod_sexo_pessoa'] == 2
mask_masculino = df['p.cod_sexo_pessoa'] == 1

ch06_fem = (mask_ch06 & mask_feminino).sum()
ch06_mas = (mask_ch06 & mask_masculino).sum()

print(f'Mulheres analfabetas 15+ : {ch06_fem}  ({(ch06_fem/total_ch06*100).round(1)}%)')
print(f'Homens analfabetos 15+   : {ch06_mas}  ({(ch06_mas/total_ch06*100).round(1)}%)')

In [ ]:
# trajetória escolar
# p.ind_frequenta_escola_memb: 1=rede pública, 2=rede particular, 3=já frequentou, 4=nunca frequentou
mask_nunca    = df['p.ind_frequenta_escola_memb'] == 4
mask_publica  = df['p.ind_frequenta_escola_memb'] == 1
mask_part     = df['p.ind_frequenta_escola_memb'] == 2
mask_ja_freq  = df['p.ind_frequenta_escola_memb'] == 3

print(f'Nunca foram à escola                         : {(mask_ch06 & mask_nunca).sum()}')
print(f'Foram à rede pública e saíram analfabetos    : {(mask_ch06 & mask_publica).sum()}')
print(f'Foram à rede particular e saíram analfabetos : {(mask_ch06 & mask_part).sum()}')
print(f'Frequentaram mas saíram sem aprender         : {(mask_ch06 & mask_ja_freq).sum()}')

In [ ]:
# distribuição territorial — proxy via d.nom_localidade_fam
# nota: matching fuzzy com loteamentosregiao.xls executado em sessão analítica separada
ch06_por_local = (
    df[mask_ch06]['d.nom_localidade_fam']
    .value_counts()
    .reset_index()
)
ch06_por_local.columns = ['localidade', 'qtd_analfabetos']

print('Top 10 localidades por analfabetismo:')
print(ch06_por_local.head(10).to_string(index=False))

In [ ]:
# cruzamentos de vulnerabilidade

# P02: analfabetos 15+ em famílias com renda zero
mask_renda_zero = df['d.vlr_renda_media_fam'] == 0
p02 = (mask_ch06 & mask_renda_zero).sum()
print(f'P02 — analfabetos 15+ em famílias com renda zero          : {p02}  ({(p02/total_ch06*100).round(1)}%)')

# P03: idosos analfabetos com renda ≤ ½ SM
mask_renda_baixa = df['d.vlr_renda_media_fam'] <= MEIO_SM
mask_idoso_analfabeto = mask_60mais & mask_analfabeto
p03 = (mask_idoso_analfabeto & mask_renda_baixa).sum()
print(f'P03 — idosos analfabetos com renda ≤ ½ SM                 : {p03}  ({(p03/ch06_idosos.sum()*100).round(1)}%)')

# P03+: idosos analfabetos, pobres, sem BPC nem Bolsa Família
mask_nao_pbf = df['d.marc_pbf'] == 0
p03_plus = (mask_idoso_analfabeto & mask_renda_baixa & mask_nao_pbf).sum()
print(f'P03+ — idosos analfabetos, pobres, sem BPC/BF (núcleo)    : {p03_plus}')

## 7. Síntese — Placar Fase 1 MVP

In [ ]:
# resumo consolidado dos indicadores calculados
resultados = {
    'RT_01': f'{rt01}% das famílias com renda per capita ≤ ½ SM',
    'RT_04': f'{rt04}% das famílias ({familias_rt04} famílias) — renda ≤ ½ SM + idoso',
    'CH_06': f'{ch06}% das pessoas 15+ são analfabetas ({total_ch06} pessoas)',
    'CH_05': 'PENDENTE',
    'CH_07': 'PENDENTE'
}

print('=== PLACAR IVS-H — FASE 1 MVP ===')
print(f'Período de referência: {PERIODO_REFERENCIA}')
print(f'Base: CadÚnico dez/2025 — {df.shape[0]} pessoas')
print()
for var, resultado in resultados.items():
    status = '✅' if resultado != 'PENDENTE' else '⏳'
    print(f'{status}  {var}: {resultado}')